# [BERTopic Learning: Update Topics](https://maartengr.github.io/BERTopic/getting_started/topicreduction/topicreduction.html)

## Topic Reduction

BERTopic uses HDBSCAN for clustering the data and it cannot specify the number of clusters you would want. To a certain extent, this is an advantage, as we can trust HDBSCAN to be better in finding the number of clusters than we are. Instead, we can try to reduce the number of topics that have been created. Below, you will find three methods of doing so.

### Manual Topic Reduction

Each resulting topic has its feature vector constructed from c-TF-IDF. Using those feature vectors, we can find the most similar topics and merge them. If we do this iteratively, starting from the least frequent topic, we can reduce the number of topics quite easily. We do this until we reach the value of nr_topics:

In [1]:
from sklearn.datasets import fetch_20newsgroups
from sentence_transformers import SentenceTransformer
from bertopic import BERTopic
from umap import UMAP
import datamapplot
import numpy as np

In [2]:
# Prepare embeddings
docs = fetch_20newsgroups(subset='all',  remove=('headers', 'footers', 'quotes'))['data']
sentence_model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = sentence_model.encode(docs, show_progress_bar=False)

topic_model = BERTopic(nr_topics=20)
topics, probs = topic_model.fit_transform(docs, embeddings)

OMP: Info #276: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitl

In [3]:
topic_model.visualize_topics()

It is also possible to manually select certain topics that you believe should be merged. For example, if topic 1 is 1_space_launch_moon_nasa and topic 2 is 2_spacecraft_solar_space_orbit it might make sense to merge those two topics:

In [4]:
topics_to_merge = [1, 2]
topic_model.merge_topics(docs, topics_to_merge)

In [5]:
topic_model.visualize_topics()

If you have several groups of topics you want to merge, create a list of lists instead:

In [9]:
topics_to_merge = [[8, 10],
                   [3, 4]]
topic_model.merge_topics(docs, topics_to_merge)

In [10]:
topic_model.visualize_topics()

### Automatic Topic Reduction

One issue with the approach above is that it will merge topics regardless of whether they are very similar. They are simply the most similar out of all options. This can be resolved by reducing the number of topics automatically. To do this, we can use HDBSCAN to cluster our topics using each c-TF-IDF representation. Then, we merge topics that are clustered together. Another benefit of HDBSCAN is that it generates outliers. These outliers prevent topics from being merged if no other topics are similar.

To use this option, we simply set nr_topics to "auto":

In [11]:
topic_model = BERTopic(nr_topics="auto")
topics, probs = topic_model.fit_transform(docs, embeddings)
topic_model.visualize_topics()

### Topic Reduction after Training

Finally, we can also reduce the number of topics after having trained a BERTopic model. The advantage of doing so is that you can decide the number of topics after knowing how many are created. It is difficult to predict before training your model how many topics that are in your documents and how many will be extracted. Instead, we can decide afterward how many topics seem realistic:

In [1]:
from bertopic import BERTopic
from sklearn.datasets import fetch_20newsgroups

# Create topics -> Typically over 50 topics
docs = fetch_20newsgroups(subset='all',  remove=('headers', 'footers', 'quotes'))['data']
topic_model = BERTopic()
topics, probs = topic_model.fit_transform(docs)

# Further reduce topics
topic_model.reduce_topics(docs, nr_topics=30)

# Access updated topics
topics = topic_model.topics_


OMP: Info #276: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitl

In [2]:
topics


[0,
 -1,
 6,
 7,
 7,
 -1,
 -1,
 0,
 0,
 -1,
 18,
 -1,
 -1,
 18,
 -1,
 6,
 7,
 1,
 1,
 9,
 9,
 2,
 -1,
 9,
 0,
 10,
 1,
 -1,
 -1,
 12,
 -1,
 1,
 2,
 0,
 13,
 2,
 -1,
 1,
 9,
 -1,
 10,
 15,
 17,
 11,
 0,
 -1,
 -1,
 3,
 1,
 -1,
 -1,
 4,
 -1,
 1,
 1,
 25,
 -1,
 -1,
 5,
 9,
 0,
 -1,
 -1,
 -1,
 -1,
 22,
 0,
 -1,
 -1,
 -1,
 3,
 -1,
 4,
 11,
 -1,
 -1,
 0,
 2,
 11,
 0,
 9,
 4,
 -1,
 17,
 3,
 8,
 -1,
 6,
 -1,
 8,
 0,
 5,
 8,
 3,
 6,
 3,
 7,
 -1,
 1,
 -1,
 27,
 -1,
 -1,
 6,
 5,
 0,
 -1,
 -1,
 4,
 2,
 2,
 -1,
 10,
 2,
 -1,
 6,
 -1,
 22,
 0,
 -1,
 -1,
 24,
 -1,
 -1,
 1,
 4,
 -1,
 5,
 -1,
 -1,
 5,
 8,
 -1,
 0,
 7,
 5,
 7,
 1,
 7,
 1,
 -1,
 2,
 12,
 2,
 -1,
 1,
 2,
 2,
 0,
 3,
 9,
 9,
 7,
 -1,
 8,
 -1,
 -1,
 6,
 10,
 3,
 -1,
 5,
 3,
 -1,
 -1,
 -1,
 8,
 10,
 22,
 13,
 3,
 -1,
 -1,
 -1,
 11,
 -1,
 0,
 -1,
 9,
 0,
 9,
 0,
 14,
 -1,
 -1,
 5,
 -1,
 -1,
 6,
 -1,
 -1,
 -1,
 5,
 8,
 5,
 8,
 6,
 -1,
 -1,
 2,
 -1,
 -1,
 -1,
 -1,
 -1,
 4,
 11,
 -1,
 7,
 8,
 9,
 2,
 -1,
 4,
 9,
 1,
 -1,
 -1,
 10,
 5,
 0,
 8,
 7,

## Update Topic Representation

The topics that are extracted from BERTopic are represented by words. These words are extracted from the documents occupying their topics using a class-based TF-IDF. This allows us to extract words that are interesting to a topic but less so to another.



### Update Topic Representation after Traning

When you have trained a model and viewed the topics and the words that represent them, you might not be satisfied with the representation. Perhaps you forgot to remove stop_words or you want to try out a different n_gram_range. We can use the function update_topics to update the topic representation with new parameters for c-TF-IDF:

In [3]:
from bertopic import BERTopic
from sklearn.datasets import fetch_20newsgroups

# Create topics
docs = fetch_20newsgroups(subset='all',  remove=('headers', 'footers', 'quotes'))['data']
topic_model = BERTopic(n_gram_range=(2, 3))
topics, probs = topic_model.fit_transform(docs)

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The 

From the model created above, one of the most frequent topics is the following:

In [4]:
topic_model.get_topic(31)[:10]

[('this program', 0.010977917684223147),
 ('24 bits', 0.007620796869112509),
 ('read menu', 0.007430926794696568),
 ('to convert', 0.006848211867155565),
 ('in read', 0.006618135246809149),
 ('display type', 0.006618135246809149),
 ('program will', 0.006015717297021165),
 ('in read menu', 0.005787550886089187),
 ('if you', 0.005099090955366716),
 ('15 16 24', 0.004936179378468856)]

Although there does seems to be some relation between words, it is difficult, at least for me, to intuitively understand what the topic is about. Instead, let's simplify the topic representation by setting n_gram_range to (1, 3) to also allow for single words.

In [5]:
topic_model.update_topics(docs, n_gram_range=(1, 3))
topic_model.get_topic(31)[:10]

[('format', 0.011284165953649533),
 ('gif', 0.010992337135095186),
 ('files', 0.010948920540704345),
 ('program', 0.00943842968124182),
 ('file', 0.008606741023130153),
 ('image', 0.00813642562089659),
 ('this program', 0.007870187925516392),
 ('menu', 0.007317473419177706),
 ('convert', 0.007253363586159782),
 ('pressing', 0.006725616510784082)]

To me, the combination of the words above seem a bit more intuitive than the words we previously had! You can play around with n_gram_range or use your own custom sklearn.feature_extraction.text.CountVectorizer and pass that
instead:

In [6]:
from sklearn.feature_extraction.text import CountVectorizer
# use customized vectorizer model
vectorizer_model = CountVectorizer(stop_words="english", ngram_range=(1, 5))
topic_model.update_topics(docs, vectorizer_model=vectorizer_model)

In [11]:
topic_model.get_topic(2)[:10]

[('hi lets', 0.1403190616350059),
 ('hi ken huh ignore', 0.1403190616350059),
 ('ignore hi', 0.1403190616350059),
 ('ites yep', 0.1403190616350059),
 ('ignore hi lets', 0.1403190616350059),
 ('ken huh', 0.1403190616350059),
 ('ignore hi lets forget hello', 0.1403190616350059),
 ('idjits hello good cheek', 0.1403190616350059),
 ('lets forget hello ites yep', 0.1403190616350059),
 ('idjits hello', 0.1403190616350059)]

### Custom Labels

The topic labels are currently automatically generated by taking the top 3 words and combining them using the _ separator. Although this is an informative label, in practice, this is definitely not the prettiest nor necessarily the most accurate label. For example, although the topic label 1_space_nasa_orbit is informative, but we would prefer to have a bit more intuitive label, such as space travel. The difficulty with creating such topic labels is that much of the interpretation is left to the user. Would space travel be more accurate or perhaps space explorations? To truly understand which labels are most suited, going into some of the documents in topics is especially helpful.

Although we can go through every single topic ourselves and try to label them, we can start by creating an overview of labels that have the length and number of words that we are looking for. To do so, we can generate our list of topic labels with .generate_topic_labels and define the number of words, the separator, word length, etc:

In [12]:
topic_labels = topic_model.generate_topic_labels(nr_words=3,
                                                 topic_prefix=False,
                                                 word_length=10,
                                                 separator=", ")


In [13]:
topic_labels

['like, use, know',
 'game, team, games',
 'key, clipper, chip',
 'hi lets, hi ken huh, ignore hi',
 'israel, israeli, jews',
 'fbi, koresh, gas',
 'car, cars, ford',
 'post, ted, frank',
 'amp, audio, condition',
 'card, drivers, diamond',
 'space, launch, station',
 'printer, print, hp',
 'health, cancer, tobacco',
 'atheists, atheism, god',
 'drive, drives, disk',
 'modem, port, serial',
 'medical, patients, hiv',
 'armenian, armenians, turkish',
 'window, widget, event',
 'gun, guns, deaths',
 'moral, morality, objective',
 'dos, pom, cds',
 'bike, bikes, miles',
 'drive, drives, disks',
 'monitor, monitors, vga',
 'migraine, drug, cancer',
 '3d, machines, contact',
 'windows, os2, nt',
 'windows, dos, mbytes',
 'islam, quran, islamic',
 'games, joystick, sega',
 'modem, fax, modems',
 'format, gif, files',
 'ham, radio, interferen',
 'error, symbol, undefined',
 'homosexual, homosexual, homosexual',
 'simms, simm, ram',
 'windows, memory, dos',
 'tax, taxes, income',
 'death pena,

In the above example, 1_space_nasa_orbit would turn into space, nasa, orbit since we selected 3 words, no topic prefix, and the , separator. We can then either change our topic_labels to whatever we want or directly pass them to .set_topic_labels so that they can be used across most visualization functions:

In [14]:
topic_model.set_topic_labels(topic_labels)

It is also possible to only change a few topic labels at a time by passing a dictionary where the key represents the topic ID and the value is the topic label:

In [15]:
topic_model.set_topic_labels({1: "Space Travel", 7: "Religion"})

Then, to make use of those custom topic labels across visualizations, such as .visualize_hierarchy(), we can use the custom_labels=True parameter that is found in most visualizations.

In [17]:
fig = topic_model.visualize_barchart(custom_labels=True)
fig

### Optimize labels

The great advantage of passing custom labels to BERTopic is that when more accurate zero-shot are released, we can simply use those on top of BERTopic to further fine-tune the labeling. For example, let's say you have a set of potential topic labels that you want to use instead of the ones generated by BERTopic. You could use the bart-large-mnli model to find which user-defined labels best represent the BERTopic-generated labels:

In [18]:
from transformers import pipeline
classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

# A selected topic representation
# 'god jesus atheists atheism belief atheist believe exist beliefs existence'
sequence_to_classify =  " ".join([word for word, _ in topic_model.get_topic(1)])

# Our set of potential topic labels
candidate_labels = ['cooking', 'dancing', 'religion']
classifier(sequence_to_classify, candidate_labels)

#{'labels': ['cooking', 'dancing', 'religion'],
# 'scores': [0.086, 0.063, 0.850],
# 'sequence': 'god jesus atheists atheism belief atheist believe exist beliefs existence'}


config.json:   0%|          | 0.00/1.15k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Device set to use mps:0


{'sequence': 'key clipper chip encryption keys government escrow nsa algorithm clipper chip',
 'labels': ['cooking', 'dancing', 'religion'],
 'scores': [0.5193642973899841, 0.34929534792900085, 0.13134030997753143]}

## Outlier reduction

When using HDBSCAN, DBSCAN, or OPTICS, a number of outlier documents might be created that do not fall within any of the created topics. These are labeled as -1. Depending on your use case, you might want to decrease the number of documents that are labeled as outliers. Fortunately, there are a number of strategies one might use to reduce the number of outliers after you have trained your BERTopic model.

The main way to reduce your outliers in BERTopic is by using the .reduce_outliers function. To make it work without too much tweaking, you will only need to pass the docs and their corresponding topics. You can pass outlier and non-outlier documents together since it will only try to reduce outlier documents and label them to a non-outlier topic.

The following is a minimal example:

In [19]:
from bertopic import BERTopic

# Train your BERTopic model
topic_model = BERTopic()
topics, probs = topic_model.fit_transform(docs)

# Reduce outliers
new_topics = topic_model.reduce_outliers(docs, topics)

### Strategies 
The default method for reducing outliers is by calculating the c-TF-IDF representations of outlier documents and assigning them to the best matching c-TF-IDF representations of non-outlier topics.

However, there are a number of other strategies one can use, either separately or in conjunction that are worthwhile to explore:

- Using the topic-document probabilities to assign topics

- Using the topic-document distributions to assign topics

- Using c-TF-IDF representations to assign topics

- Using document and topic embeddings to assign topics

### Probabilities

This strategy uses the soft-clustering as performed by HDBSCAN to find the best matching topic for each outlier document. To use this, make sure to calculate the probabilities beforehand by instantiating BERTopic with calculate_probabilities=True.

In [22]:
from bertopic import BERTopic

# Train your BERTopic model and calculate the document-topic probabilities
topic_model = BERTopic(calculate_probabilities=True)
topics, probs = topic_model.fit_transform(docs)

# Reduce outliers using the `probabilities` strategy
new_topics = topic_model.reduce_outliers(docs, topics, probabilities=probs, strategy="probabilities")


In [31]:
print(f"There are topics left: {len(set(new_topics))}")

There are topics left: 213


### Topic Distribution

Use the topic distributions, as calculated with .approximate_distribution to find the most frequent topic in each outlier document. You can use the distributions_params variable to tweak the parameters of .approximate_distribution.

In [32]:
from bertopic import BERTopic

# Train your BERTopic model
topic_model = BERTopic()
topics, probs = topic_model.fit_transform(docs)

# Reduce outliers using the `distributions` strategy
new_topics = topic_model.reduce_outliers(docs, topics, strategy="distributions")
print(f"There are topics left: {len(set(new_topics))}")

There are topics left: 215


### c-TF-IDF 

Calculate the c-TF-IDF representation for each outlier document and find the best matching c-TF-IDF topic representation using cosine similarity.

In [33]:
from bertopic import BERTopic

# Train your BERTopic model
topic_model = BERTopic()
topics, probs = topic_model.fit_transform(docs)

# Reduce outliers using the `c-tf-idf` strategy
new_topics = topic_model.reduce_outliers(docs, topics, strategy="c-tf-idf")
print(f"There are topics left: {len(set(new_topics))}")

There are topics left: 233


### Embeddings

Using the embeddings of each outlier documents, find the best matching topic embedding using cosine similarity.

In [34]:
from bertopic import BERTopic

# Train your BERTopic model
topic_model = BERTopic()
topics, probs = topic_model.fit_transform(docs)

# Reduce outliers using the `embeddings` strategy
new_topics = topic_model.reduce_outliers(docs, topics, strategy="embeddings")
print(f"There are topics left: {len(set(new_topics))}")

There are topics left: 220


### Chain Strategies

Since the .reduce_outliers function does not internally update the topics, we can easily try out different strategies but also chain them together. You might want to do a first pass with the "c-tf-idf" strategy as it is quite fast. Then, we can perform the "distributions" strategy on the outliers that are left since this method is typically much slower:

In [35]:
# Use the "c-TF-IDF" strategy with a threshold
new_topics = topic_model.reduce_outliers(docs, topics , strategy="c-tf-idf", threshold=0.1)

# Reduce all outliers that are left with the "distributions" strategy
new_topics = topic_model.reduce_outliers(docs, new_topics, strategy="distributions")
print(f"There are topics left: {len(set(new_topics))}")

There are topics left: 220


### Update Topics

After generating our updated topics, we can feed them back into BERTopic in one of two ways. We can either update the topic representations themselves based on the documents that now belong to new topics or we can only update the topic frequency without updating the topic representations themselves.

### Update Topic Representation

When outlier documents are generated, they are not used when modeling the topic representations. These documents are completely ignored when finding good descriptions of topics. Thus, after having reduced the number of outliers in your topic model, you might want to update the topic representations with the documents that now belong to actual topics. To do so, we can make use of the .update_topics function:

In [36]:
topic_model.update_topics(docs, topics=new_topics)

2025-03-10 11:50:16,733 - BERTopic - WARNING: Using a custom list of topic assignments may lead to errors if topic reduction techniques are used afterwards. Make sure that manually assigning topics is the last step in the pipeline.Note that topic embeddings will also be created through weightedc-TF-IDF embeddings instead of centroid embeddings.


As seen above, you will only need to pass the documents on which the model was trained including the new topics that were generated using one of the above four strategies.

### Exploration
When you are reducing the number of topics, it might be worthwhile to iteratively visualize the results in order to get an intuitive understanding of the effect of the above four strategies. Making use of .visualize_documents, we can quickly iterate over the different strategies and view their effects. Here, an example will be shown on how to approach such a pipeline.

First, we train our model:

In [2]:
from umap import UMAP
from bertopic import BERTopic
from sklearn.datasets import fetch_20newsgroups
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer

# Prepare data, extract embeddings, and prepare sub-models
docs = fetch_20newsgroups(subset='all',  remove=('headers', 'footers', 'quotes'))['data']
umap_model = UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric='cosine', random_state=42)
vectorizer_model = CountVectorizer(stop_words="english")
sentence_model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = sentence_model.encode(docs, show_progress_bar=True)

# We reduce our embeddings to 2D as it will allows us to quickly iterate later on
reduced_embeddings = UMAP(n_neighbors=10, n_components=2, 
                          min_dist=0.0, metric='cosine').fit_transform(embeddings)

# Train our topic model
topic_model = BERTopic(embedding_model=sentence_model, umap_model=umap_model, 
                       vectorizer_model=vectorizer_model, calculate_probabilities=True, nr_topics=40)
topics, probs = topic_model.fit_transform(docs, embeddings)


Batches:   0%|          | 0/589 [00:00<?, ?it/s]

OMP: Info #276: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitl

After having trained our model, let us take a look at the 2D representation of the generated topics:

In [3]:
topic_model.visualize_documents(docs, reduced_embeddings=reduced_embeddings, 
                                hide_document_hover=True, hide_annotations=True)


Next, we reduce the number of outliers using the probabilities strategy:

In [5]:
new_topics = topic_model.reduce_outliers(docs, topics, probabilities=probs, 
                             threshold=0.05, strategy="probabilities")
topic_model.update_topics(docs, topics=new_topics)

2025-03-10 13:54:50,455 - BERTopic - WARNING: Using a custom list of topic assignments may lead to errors if topic reduction techniques are used afterwards. Make sure that manually assigning topics is the last step in the pipeline.Note that topic embeddings will also be created through weightedc-TF-IDF embeddings instead of centroid embeddings.


And finally, we visualize the results:

In [6]:
topic_model.visualize_documents(docs, reduced_embeddings=reduced_embeddings, 
                                hide_document_hover=True, hide_annotations=True)
